In [1]:
import os
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import load_img, img_to_array

In [2]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, classification_report

In [12]:
EPIC_MODEL_PATH = "models/epic_detector.keras"
SOURCE_DIR = "data/img/img_align_celeba"
ATTR_PATH = "data/list_attr_celeba.csv"  # ändra till din fil
REVIEW_EPIC_DIR = "data/review_epic_sort/epic_high_score"
REVIEW_THIN_DIR = "data/review_epic_sort/thin_low_score"

os.makedirs(REVIEW_EPIC_DIR, exist_ok=True)
os.makedirs(REVIEW_THIN_DIR, exist_ok=True)

IMG_SIZE = (128, 128)  # gamla epic-modellens storlek


epic_model = load_model(EPIC_MODEL_PATH)
attrs = pd.read_csv(ATTR_PATH)

In [13]:
mustache_pool = attrs[attrs["Mustache"] == 1].copy()

print("CelebA mustache-bilder:", len(mustache_pool))

CelebA mustache-bilder: 8417


In [11]:
import glob

matches = glob.glob("**/000109.jpg", recursive=True)
print(matches[:10])

['data/img/img_align_celeba/000109.jpg', 'data/dataset_v3/mustache/000109.jpg']


In [14]:
already_used = set()

for folder in [
    "data/epic_dataset/epic",
    "data/epic_dataset/thin",
    "data/epic_dataset_v2/epic",
    "data/epic_dataset_v2/thin",
]:
    if os.path.exists(folder):
        already_used |= set(os.listdir(folder))

mustache_pool = mustache_pool[~mustache_pool["image_id"].isin(already_used)]

print("Kvar att söka i:", len(mustache_pool))

Kvar att söka i: 8417


In [15]:
rows = []

for i, fname in enumerate(mustache_pool["image_id"]):
    path = os.path.join(SOURCE_DIR, fname)

    if not os.path.exists(path):
        continue

    img = load_img(path, target_size=IMG_SIZE)
    arr = img_to_array(img)
    arr = np.expand_dims(arr, axis=0)

    score = float(epic_model.predict(arr, verbose=0)[0][0])

    rows.append({
        "filename": fname,
        "path": path,
        "epic_score": score
    })

    if (i + 1) % 500 == 0:
        print(f"{i+1}/{len(mustache_pool)} klara")

epic_search_df = pd.DataFrame(rows)

500/8417 klara
1000/8417 klara
1500/8417 klara
2000/8417 klara
2500/8417 klara
3000/8417 klara
3500/8417 klara
4000/8417 klara
4500/8417 klara
5000/8417 klara
5500/8417 klara
6000/8417 klara
6500/8417 klara
7000/8417 klara
7500/8417 klara
8000/8417 klara


In [16]:
epic_search_df = epic_search_df.sort_values("epic_score", ascending=False)

epic_search_df.head(50)

,filename,path,epic_score
3584,089709.jpg,data/img/img_align_celeba/089709.jpg,0.997866
4442,110452.jpg,data/img/img_align_celeba/110452.jpg,0.997771
2878,071197.jpg,data/img/img_align_celeba/071197.jpg,0.995698
5616,138079.jpg,data/img/img_align_celeba/138079.jpg,0.994948
5143,126817.jpg,data/img/img_align_celeba/126817.jpg,0.994163
414,011068.jpg,data/img/img_align_celeba/011068.jpg,0.992786
328,008571.jpg,data/img/img_align_celeba/008571.jpg,0.992677
2392,059187.jpg,data/img/img_align_celeba/059187.jpg,0.991568
1966,048218.jpg,data/img/img_align_celeba/048218.jpg,0.986794
2031,049992.jpg,data/img/img_align_celeba/049992.jpg,0.986620


In [17]:
EPIC_TOP_N = 2000
THIN_TOP_N = 2000

epic_sorted = epic_search_df.sort_values("epic_score", ascending=False)
thin_sorted = epic_search_df.sort_values("epic_score", ascending=True)

epic_candidates = epic_sorted.head(EPIC_TOP_N)
thin_candidates = thin_sorted.head(THIN_TOP_N)

In [18]:
for _, row in epic_candidates.iterrows():
    src = row["path"]
    dst = os.path.join(
        REVIEW_EPIC_DIR,
        f"{row['epic_score']:.3f}_{row['filename']}"
    )

    if os.path.exists(src):
        shutil.copy2(src, dst)

for _, row in thin_candidates.iterrows():
    src = row["path"]
    dst = os.path.join(
        REVIEW_THIN_DIR,
        f"{row['epic_score']:.3f}_{row['filename']}"
    )

    if os.path.exists(src):
        shutil.copy2(src, dst)

print(f"Kopierade {len(epic_candidates)} high-score bilder till epic-review")
print(f"Kopierade {len(thin_candidates)} low-score bilder till thin-review")

Kopierade 2000 high-score bilder till epic-review
Kopierade 2000 low-score bilder till thin-review


In [19]:
import os
import shutil
import numpy as np
import pandas as pd
from tensorflow.keras.utils import load_img, img_to_array

SOURCE_REVIEW_DIR = "data/review_epic_sort/epic_high_score"
OUT_THIN_DIR = "data/review_epic_sort/thin_from_epic_high_score_top1000"

IMG_SIZE = (128, 128)  # gamla epic-modellens input-size

os.makedirs(OUT_THIN_DIR, exist_ok=True)

rows = []

files = [
    f for f in os.listdir(SOURCE_REVIEW_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

for i, fname in enumerate(files):
    path = os.path.join(SOURCE_REVIEW_DIR, fname)

    img = load_img(path, target_size=IMG_SIZE)
    arr = img_to_array(img)
    arr = np.expand_dims(arr, axis=0)

    raw_score = float(epic_model.predict(arr, verbose=0)[0][0])

    rows.append({
        "filename": fname,
        "path": path,
        "raw_score": raw_score
    })

    if (i + 1) % 500 == 0:
        print(f"{i+1}/{len(files)} klara")

df_epic_high_rescored = pd.DataFrame(rows)

# Om raw_score nära 1 verkar betyda THIN:
thin_candidates = df_epic_high_rescored.sort_values(
    "raw_score",
    ascending=False
).head(1000)

for _, row in thin_candidates.iterrows():
    dst = os.path.join(
        OUT_THIN_DIR,
        f"{row['raw_score']:.3f}_{row['filename']}"
    )
    shutil.copy2(row["path"], dst)

print(f"Kopierade {len(thin_candidates)} mest thin-kandidater till {OUT_THIN_DIR}")

500/1935 klara
1000/1935 klara
1500/1935 klara
Kopierade 1000 mest thin-kandidater till data/review_epic_sort/thin_from_epic_high_score_top1000


In [20]:
import os
import shutil
import numpy as np
import pandas as pd

from tensorflow.keras.utils import load_img, img_to_array

SOURCE_REVIEW_DIR = "data/review_epic_sort/thin_low_score"
OUT_EPIC_DIR = "data/review_epic_sort/epic_from_thin_low_score_top1000"

IMG_SIZE = (128, 128)  # gamla epic-modellens input-size

os.makedirs(OUT_EPIC_DIR, exist_ok=True)

rows = []

files = [
    f for f in os.listdir(SOURCE_REVIEW_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

print(f"Hittade {len(files)} bilder")

for i, fname in enumerate(files):
    path = os.path.join(SOURCE_REVIEW_DIR, fname)

    try:
        img = load_img(path, target_size=IMG_SIZE)
        arr = img_to_array(img)
        arr = np.expand_dims(arr, axis=0)

        raw_score = float(
            epic_model.predict(arr, verbose=0)[0][0]
        )

        rows.append({
            "filename": fname,
            "path": path,
            "raw_score": raw_score
        })

        if (i + 1) % 500 == 0:
            print(f"{i+1}/{len(files)} klara")

    except Exception as e:
        print(f"Fel på {fname}: {e}")

df_rescored = pd.DataFrame(rows)

print(f"\nScorade {len(df_rescored)} bilder")

# Lägre score verkar motsvara tjockare/mer epic mustascher
epic_candidates = (
    df_rescored
    .sort_values("raw_score", ascending=True)
    .head(1000)
)

for _, row in epic_candidates.iterrows():
    dst = os.path.join(
        OUT_EPIC_DIR,
        f"{row['raw_score']:.3f}_{row['filename']}"
    )

    shutil.copy2(row["path"], dst)

print(f"\nKopierade {len(epic_candidates)} mest epic-kandidater")
print(f"Sparade i: {OUT_EPIC_DIR}")

print("\nTopp 20 mest epic enligt modellen:")
display(epic_candidates.head(20))

Hittade 1474 bilder
500/1474 klara
1000/1474 klara

Scorade 1474 bilder

Kopierade 1000 mest epic-kandidater
Sparade i: data/review_epic_sort/epic_from_thin_low_score_top1000

Topp 20 mest epic enligt modellen:


,filename,path,raw_score
253,0.000_140646.jpg,data/review_epic_sort/thin_low_score/0.000_140...,0.000363
559,0.000_022843.jpg,data/review_epic_sort/thin_low_score/0.000_022...,0.000421
49,0.000_017491.jpg,data/review_epic_sort/thin_low_score/0.000_017...,0.000461
622,0.001_034069.jpg,data/review_epic_sort/thin_low_score/0.001_034...,0.000526
1262,0.001_027865.jpg,data/review_epic_sort/thin_low_score/0.001_027...,0.000537
681,0.001_115461.jpg,data/review_epic_sort/thin_low_score/0.001_115...,0.000780
544,0.001_117880.jpg,data/review_epic_sort/thin_low_score/0.001_117...,0.000846
247,0.001_120323.jpg,data/review_epic_sort/thin_low_score/0.001_120...,0.000874
426,0.001_167179.jpg,data/review_epic_sort/thin_low_score/0.001_167...,0.001022
664,0.001_179960.jpg,data/review_epic_sort/thin_low_score/0.001_179...,0.001028


In [ ]:
epic_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history1 = epic_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop]
)

In [3]:
DATA_DIR = "data/epic_dataset"
MODEL_OUT = "models/epic_detector2_178.keras"

IMG_SIZE = (178, 178)
BATCH_SIZE = 32
SEED = 42

In [4]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

print("Klasser:", train_ds.class_names)

Found 2128 files belonging to 2 classes.
Using 1703 files for training.
Found 2128 files belonging to 2 classes.
Using 425 files for validation.
Klasser: ['epic', 'thin']


In [5]:
def build_epic_model():
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.03),
        tf.keras.layers.RandomZoom(0.05),
        tf.keras.layers.RandomContrast(0.10),
    ])

    model = tf.keras.Sequential([
        data_augmentation,
        tf.keras.layers.Rescaling(1./255),

        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(64, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(128, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(256, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    return model

In [6]:
epic_model = build_epic_model()

In [7]:
epic_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

In [8]:
history = epic_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=35,
    callbacks=[early_stop]
)

Epoch 1/35
54/54 ━━━━━━━━━━━━━━━━━━━━ 22s 375ms/step - accuracy: 0.6418 - loss: 0.6325 - precision: 0.6511 - recall: 0.7180 - val_accuracy: 0.6847 - val_loss: 0.5597 - val_precision: 0.7603 - val_recall: 0.7077
Epoch 2/35
54/54 ━━━━━━━━━━━━━━━━━━━━ 20s 370ms/step - accuracy: 0.7839 - loss: 0.4782 - precision: 0.8049 - recall: 0.7891 - val_accuracy: 0.6682 - val_loss: 0.6520 - val_precision: 0.9685 - val_recall: 0.4731
Epoch 3/35
54/54 ━━━━━━━━━━━━━━━━━━━━ 21s 383ms/step - accuracy: 0.8761 - loss: 0.2998 - precision: 0.8818 - recall: 0.8885 - val_accuracy: 0.9153 - val_loss: 0.2136 - val_precision: 0.9444 - val_recall: 0.9154
Epoch 4/35
54/54 ━━━━━━━━━━━━━━━━━━━━ 21s 383ms/step - accuracy: 0.9049 - loss: 0.2293 - precision: 0.9115 - recall: 0.9115 - val_accuracy: 0.9294 - val_loss: 0.1783 - val_precision: 0.9713 - val_recall: 0.9115
Epoch 5/35
54/54 ━━━━━━━━━━━━━━━━━━━━ 20s 377ms/step - accuracy: 0.9213 - loss: 0.2018 - precision: 0.9305 - recall: 0.9224 - val_accuracy: 0.9435 - val_los

In [9]:
epic_model.save("models/epic_detector_v2_95.keras")